# Knowledge Graph Creation

In [1]:
import json
import sys
import pandas as pd
import collections 
import os
import numpy as np
from itertools import chain
from itertools import combinations
sys.path.insert(0, '..')
from src.experiment_utils.helper_classes import token, span, repository
from src.d02_corpus_statistics.corpus import Corpus
import types

cwd = os.getcwd()
pol_dir = cwd+"/../src/d01_data"

In [2]:
pol_df = pd.read_pickle(pol_dir+"/preprocessed_dataframe.pkl")[["Policy","Text","Tokens","Curation"]]
meta_df = pd.read_csv(pol_dir+"/EU_metadata.csv", delimiter=";")

Each policy has chapters, each chapter has sections, each section has articles
Each article has spans
Each span has text labeled with a tag, which is a specified version of a feature which is a specified version of a layer

In [3]:
from owlready2 import *

owl_path = cwd+"/auxil/ontology.owl"
onto = get_ontology(owl_path).load()

with onto:
    # classes
    #policy structure
    class Policy(Thing): 
        pass
    class Chapter(Thing): 
        pass
    class Section(Thing): 
        pass
    class Article(Thing): 
        pass
    #spans
    class Layer(Thing): 
        pass
    class Feature(Thing): 
        pass
    class Tag(Thing): 
        pass
    class Span(Thing): 
        pass

    # object properties
    #structure
    class hasChapter(ObjectProperty):
        domain = [Policy]
        range = [Chapter]
    class hasSection(ObjectProperty):
        domain = [Chapter]
        range = [Section]
    class hasArticle(ObjectProperty):
        domain = [Section]
        range = [Article]
    class hasSpan(ObjectProperty):
        domain = [Article]
        range = [Span]
    class isChapterOf(ObjectProperty):
        domain = [Chapter]
        range = [Policy]
        inverse_property = hasChapter
    class isSectionOf(ObjectProperty):
        domain = [Section]
        range = [Chapter]
        inverse_property = hasSection
    class isArticleOf(ObjectProperty):
        domain = [Article]
        range = [Section]
        inverse_property = hasArticle
    class isSpanOf(ObjectProperty):
        domain = [Span]
        range = [Article]
        inverse_property = hasSpan
    #spans
    class inTag(ObjectProperty):
        domain = [Span]
        range = [Tag]
    class TagOf(ObjectProperty):
        domain = [Tag]
        range = [Span]
        inverse_property = inTag
    ### ??????????????
    ### do i keep these here or specify strict elsewhere?
    ### since tags always have the same feature which always have the same layer
    class inLayer(ObjectProperty):
        domain = [Feature]
        range = [Layer]
    class inFeature(ObjectProperty):
        domain = [Tag]
        range = [Feature]
    class LayerOf(ObjectProperty):
        domain = [Layer]
        range = [Feature]
        inverse_property = inLayer
    class FeatureOf(ObjectProperty):
        domain = [Feature]
        range = [Tag]
        inverse_property = inFeature
    
    # data properties
    #policy structure
    class policy_code(DataProperty):
        domain = [Policy]
        range = [str]
    class chapter_num(DataProperty):
        domain = [Chapter]
        range = [str]
    class section_num(DataProperty):
        domain = [Section]
        range = [str]
    class article_num(DataProperty):
        domain = [Article]
        range = [str]
    #spans
    class layer_name(DataProperty):
        domain = [Layer]
        range = [str]
    class feature_name(DataProperty):
        domain = [Feature]
        range = [str]
    class tag_name(DataProperty):
        domain = [Tag]
        range = [str]
    class span_id(DataProperty):
        domain = [Span]
        range = [str]
    class span_text(DataProperty):
        domain = [Span]
        range = [str]

## KG from dataset
Each policy has articles
Each article has features, instruments and list(policy_design_tags)
Each feature has a tag with a span (or a span with a tag)

In [4]:
instrument_type_tags=[
    "VoluntaryAgrmt",
    "FrameworkPolicy",
    "TradablePermit",
    "RegulatoryInstr",
    "TaxIncentives",
    "Subsidies_Incentives",
    "RD_D",
    "PublicInvt",
    "Edu_Outreach",
    "Unspecified"
]

policy_design_tags = {
    "actor":[
        "Authority_default",
        "Authority_legislative",
        "Authority_established",
        "Authority_monitoring",
        "Addressee_default",
        "Addressee_resource",
        "Addressee_monitored",
        "Addressee_sector"
    ],
    "compliance":[
        "Form_sanctioning",
        "Form_monitoring"
    ],
    "reference":[
        "Ref_OtherPolicy",
        "Ref_PolicyAmended",
        "Ref_Strategy_Agreement"
    ],
    "objective":[
        "Objective_QuantTarget",
        "Objective_QualIntention",
        "Objective_QuantTarget_noCCM",
        "Objective_QualIntention_noCCM"
    ],
    "resource":[
        "Resource_MonSpending",
        "Resource_MonRevenues",
        "Resource_Other"
    ],
    "time":[
        "Time_PolDuration"
        "Time_Monitoring"
        "Time_Resources"
        "Time_Compliance"
        "Time_InEffect"
    ]
}

In [5]:
'''
with onto:
    # layers
    class Instrumenttypes(Layer):
        pass
    class Policydesigncharacteristics(Layer):
        pass
    # features
    class Instrumenttype(Feature):
        pass
    class Actor(Feature):
        pass
    class Compliance(Feature):
        pass
    class Reference(Feature):
        pass
    class Objective(Feature):
        pass
    class Resource(Feature):
        pass
    class Time(Feature):
        pass
    # tags
'''

'\nwith onto:\n    # layers\n    class Instrumenttypes(Layer):\n        pass\n    class Policydesigncharacteristics(Layer):\n        pass\n    # features\n    class Instrumenttype(Feature):\n        pass\n    class Actor(Feature):\n        pass\n    class Compliance(Feature):\n        pass\n    class Reference(Feature):\n        pass\n    class Objective(Feature):\n        pass\n    class Resource(Feature):\n        pass\n    class Time(Feature):\n        pass\n    # tags\n'

In [6]:
for i in Thing.instances():
    destroy_entity(i)
    #print(i)

In [7]:
# to reexamine, currently excluding:
# articles with id's [policy_code]_Whereas or [policy_code]_front
works = []
fails = []
for i in range(len(list(pol_df.index))):
    #x.append(pol_df.index[i].split("_")[3])
    try:
        pol_df.index[i].split("_")[3]
        works.append(i)
    except:
        fails.append(i)
#pol_df.iloc[fails]

In [8]:
policies = {}
chapters = {}
sections = {}
articles = {}

for ind in pol_df.index:
    deets = ind.split("_")
    policy_code = "_".join(deets[:2])
    #ignore whereas and front bits for now
    if deets[2] == "Whereas" or deets[2] == "front":
        continue
    chapter_num = deets[5]
    section_num = deets[7]
    article_num = deets[9]
    # Policy
    if policy_code not in policies:
        policy = onto.Policy(policy_code) #unique Policy instance with name of policy_code
        policy.policy_code = [policy_code] #sets the Policy instance's policy_code to the policy_code
        policies[policy_code] = policy #stores the Policy instance in the dictionary
    else:
        policy = policies[policy_code]
    # Chapter
    chapter_key = f"{policy_code}_Chapter_{chapter_num}" #makes unique chapter key
    if chapter_key not in chapters:
        chapter = onto.Chapter(chapter_key)
        chapter.chapter_num = [chapter_num]
        chapter.isChapterOf = [policy]
        chapters[chapter_key] = chapter
    else:
        chapter = chapters[chapter_key]
    # Section
    section_key = f"{policy_code}_Chapter_{chapter_num}_Section_{section_num}"
    if section_key not in sections:
        section = onto.Section(section_key)
        section.section_num = [section_num]
        section.isSectionOf = [chapter]
        sections[section_key] = section
    else:
        section = sections[section_key]
    # Article
    article_key = f"{policy_code}_Chapter_{chapter_num}_Section_{section_num}_Article_{article_num}"
    if article_key not in articles:
        article = onto.Article(article_key)
        article.article_num = [article_num]
        article.isArticleOf = [section]
        articles[article_key] = article
    else:
        article = articles[article_key]
onto.save(file=f"{cwd}/auxil/policy_kg_populated.owl", format="rdfxml")

In [ ]:
no_tag_lst = []
for ind in pol_df.index:
    deets = ind.split("_")
    policy_code = "_".join(deets[:2])
    #ignore whereas and front bits for now
    if deets[2] == "Whereas" or deets[2] == "front":
        continue
    chapter_num = deets[5]
    section_num = deets[7]
    article_num = deets[9]
    article_key = f"{policy_code}_Chapter_{chapter_num}_Section_{section_num}_Article_{article_num}"

    article = articles[article_key]
    for spanobj in pol_df.loc[ind, "Curation"]:
        span_id = f"{article_key}_{spanobj.span_id}"
        if not spanobj.tag:
            no_tag_lst.append((ind, spanobj.span_id))
            continue
        if spanobj.feature == "Technologyandapplicationspecificity":
            continue
        span = onto.Span(span_id+"_span")
        span.span_id = [span_id]
        span.span_text = [spanobj.text]
        # tag
        tag = onto.Tag(spanobj.tag)
        tag.tag_name = [spanobj.tag]
        # can i assert feature/layer inheritance globally? should i?
        feature = onto.Feature(spanobj.feature)
        feature.feature_name = [spanobj.feature]
        layer = onto.Layer(spanobj.layer)
        layer.layer_name = [spanobj.layer]
        feature.inLayer = [layer]
        tag.inFeature = [feature]
        span.inTag = [tag]
        #span.isSpanOf = [article] # is this necessary given the following line? # why isnt it showing up?
        article.hasSpan = [span]
    articles[article_key] = article

onto.save(file=f"{cwd}/auxil/policy_kg_populated.owl", format="rdfxml")

In [10]:
# to reexamine, currently excluding:
# spans labeled with the feature value "end" and tag value ""
for inid, spid in no_tag_lst:
    for spanitm in pol_df.loc[inid, "Curation"]:
        if spanitm.span_id == spid:
            #print(f"\n{inid, spid}\n{spanitm.text}\n{spanitm.tag}\n{spanitm.feature}\n{spanitm.layer}")
            #print(spanitm)
            spanitm